# Program 08: Student Success Prediction Using Learning Behavior
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Course**: MACSE502 - Python for Data Science Lab  
**Week**: 04 | **Date**: 30-07-2026 | **Type**: EP  

## Problem Statement
An online learning platform wants to study student engagement. Create a DataFrame containing StudyHours, QuizScore, AssignmentScore, AttendanceRate, InteractionCount, and CourseCompletion. Perform the following operations: create Log_StudyHours, categorize QuizScore into Needs Support, Progressing, and Excelling, generate the correlation matrix, create a Violin Plot for QuizScore grouped by CourseCompletion, and analyze the student learning patterns.

## Objectives
- To model online learner engagement using six behavioural and performance attributes.
- To engineer Log_StudyHours in order to reduce the skewness of the study effort feature.
- To band QuizScore into the three performance levels Needs Support, Progressing and Excelling.
- To identify which engagement signals are most strongly associated with course completion using a correlation matrix.
- To compare the full distribution of QuizScore between completers and non-completers by means of a violin plot.
- To translate the resulting patterns into an actionable early-warning rule for at-risk learners.


In [ ]:
# --- Cell 1: Generated Learner Engagement Dataset ---
# Load Necessary Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Fix the random seed for a reproducible cohort
np.random.seed(2026)

n = 180

# Step 2: Generate the learner engagement attributes
study_hours = np.random.lognormal(mean=3.1, sigma=0.55, size=n)
quiz_score = np.clip(np.random.normal(66, 15, n), 0, 100)
assignment_score = np.clip(quiz_score + np.random.normal(4, 9, n), 0, 100)
attendance_rate = np.clip(np.random.normal(74, 16, n), 0, 100)
interaction_count = np.random.poisson(lam=18, size=n)

# Derive CourseCompletion from a weighted engagement score plus noise
engagement = (0.030 * study_hours + 0.045 * quiz_score +
              0.030 * assignment_score + 0.025 * attendance_rate +
              0.040 * interaction_count + np.random.normal(0, 0.9, n))
completion = (engagement > np.median(engagement)).astype(int)

# Step 3: Assemble the DataFrame
df = pd.DataFrame({
    'StudyHours': study_hours.round(2),
    'QuizScore': quiz_score.round(1),
    'AssignmentScore': assignment_score.round(1),
    'AttendanceRate': attendance_rate.round(1),
    'InteractionCount': interaction_count,
    'CourseCompletion': completion
})

print("First 10 learner records")
print(df.head(10))
print("\nShape =", df.shape)
print("\nCourse completion counts:")
print(df['CourseCompletion'].value_counts())


In [ ]:
# --- Cell 2: Descriptive Statistics and Skewness of the Cohort ---
# Step 4: Summarise the cohort
print("Descriptive statistics of the learner cohort")
print(df.describe().round(2))

print("\nOverall completion rate =",
      round(df['CourseCompletion'].mean() * 100, 2), "%")
print("\nSkewness of each numeric attribute:")
print(df.drop(columns='CourseCompletion').skew().round(4))


In [ ]:
# --- Cell 3: Engineered Log_StudyHours and its Effect on Skewness ---
# Step 5: Feature engineering - logarithm of StudyHours
df['Log_StudyHours'] = np.log(df['StudyHours'])

print("Dataset after adding Log_StudyHours")
print(df[['StudyHours', 'Log_StudyHours', 'QuizScore',
          'CourseCompletion']].head(10))

print("\nSkewness of StudyHours     =", round(df['StudyHours'].skew(), 4))
print("Skewness of Log_StudyHours =", round(df['Log_StudyHours'].skew(), 4))
print("\nStudyHours range     :", df['StudyHours'].min(), "to", df['StudyHours'].max())
print("Log_StudyHours range :", round(df['Log_StudyHours'].min(), 4),
      "to", round(df['Log_StudyHours'].max(), 4))


In [ ]:
# --- Cell 4: QuizScore Banded into Three Performance Levels ---
# Step 6: Band QuizScore into three performance levels
df['QuizBand'] = pd.cut(df['QuizScore'],
                        bins=[-0.01, 50, 75, 100],
                        labels=['Needs Support', 'Progressing', 'Excelling'])

print("Dataset after banding QuizScore")
print(df[['QuizScore', 'QuizBand', 'AttendanceRate',
          'CourseCompletion']].head(10))

print("\nLearners in each performance band:")
print(df['QuizBand'].value_counts().sort_index())

print("\nQuizScore range covered by each band:")
print(df.groupby('QuizBand', observed=True)['QuizScore'].agg(['min', 'max', 'count']))


In [ ]:
# --- Cell 5: Correlation Matrix of the Engagement Signals ---
# Step 7: Correlation matrix of the numeric attributes
numeric_cols = ['StudyHours', 'Log_StudyHours', 'QuizScore', 'AssignmentScore',
                'AttendanceRate', 'InteractionCount', 'CourseCompletion']
corr = df[numeric_cols].corr()

print("Correlation matrix")
print(corr.round(3))

print("\nAssociation of each signal with CourseCompletion (strongest first):")
print(corr['CourseCompletion'].drop('CourseCompletion')
      .sort_values(ascending=False).round(4))


In [ ]:
# --- Cell 6: Violin Plot of QuizScore Grouped by Course Completion ---
# Step 8: Violin plot of QuizScore grouped by CourseCompletion
not_completed = df.loc[df['CourseCompletion'] == 0, 'QuizScore'].values
completed = df.loc[df['CourseCompletion'] == 1, 'QuizScore'].values

plt.figure(figsize=(8, 5.5))
parts = plt.violinplot([not_completed, completed],
                       positions=[1, 2], showmeans=True,
                       showmedians=True, widths=0.75)

for body, colour in zip(parts['bodies'], ['#d62728', '#2ca02c']):
    body.set_facecolor(colour)
    body.set_alpha(0.55)
    body.set_edgecolor('black')

plt.xticks([1, 2], ['Not Completed (0)', 'Completed (1)'])
plt.ylabel('QuizScore')
plt.title('Distribution of QuizScore by Course Completion')
plt.grid(True, axis='y', linestyle='--', alpha=0.35)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 7: Learning Behaviour Summary and Early-Warning Threshold ---
# Step 9: Learning patterns and an early-warning threshold
band_summary = df.groupby('QuizBand', observed=True).agg(
    Learners=('CourseCompletion', 'size'),
    Mean_StudyHours=('StudyHours', 'mean'),
    Mean_Attendance=('AttendanceRate', 'mean'),
    Mean_Interactions=('InteractionCount', 'mean'),
    Completion_Rate=('CourseCompletion', 'mean')
).round(3)

print("Learning behaviour summary by quiz performance band")
print(band_summary)

print("\nGroup means by completion status:")
print(df.groupby('CourseCompletion')[
    ['StudyHours', 'QuizScore', 'AttendanceRate', 'InteractionCount']
].mean().round(2))

risk_cut = df['QuizScore'].quantile(0.25)
at_risk = df[(df['QuizScore'] <= risk_cut) & (df['CourseCompletion'] == 0)]
print("\nLower quartile of QuizScore =", round(risk_cut, 2))
print("Learners at or below it who did not complete =", len(at_risk))


## Conclusion
This exercise programme applied feature engineering and exploratory data analysis to an online learning platform in order to understand which behaviours accompany successful course completion. A cohort of 180 learners described by six engagement attributes was generated, two features were engineered from the raw columns, and the relationship between engagement and completion was examined both numerically and graphically. The results showed that:

- StudyHours was the only clearly right skewed attribute, and the logarithmic transformation reduced that skewness so that a few very high effort learners could not dominate the analysis.
- Supplying explicit cut points of 50 and 75 to pd.cut() produced performance bands with genuine pedagogic meaning, and the cohort concentrated in the Progressing band.
- The correlation matrix exposed redundancy as well as association, since StudyHours with Log_StudyHours and QuizScore with AssignmentScore were each strongly related.
- The violin plot showed the completers' QuizScore distribution sitting distinctly higher than that of the non-completers, while the overlap through the middle range showed that quiz performance alone cannot determine completion.
- Completion rate, study hours, attendance and interaction counts all rose together across the three quiz bands, and the lower quartile of QuizScore provided a workable early-warning threshold for identifying at-risk learners.

Overall, this exercise demonstrated why a violin plot is preferable to a simple bar chart of group averages, since the overlapping region it revealed is precisely where an early-warning model would make its mistakes and would have remained invisible had only the means been compared. Combining an engineered feature, meaningful categorisation, a correlation ranking and a distributional plot produced a picture detailed enough to support a concrete intervention rule, which is exactly what a learning analytics dashboard is expected to deliver for tracking engagement and identifying learners at risk of dropping out.